<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marco-canas/5_ml_dl_g_lideres/blob/main/2_curso_formacion_ml_dl_g_lideres/3_clases_fruto_lectura_geron_deep_learning/11_chapter/1_introduccion/1_introduccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marco-canas/5_ml_dl_g_lideres/blob/main/2_curso_formacion_ml_dl_g_lideres/3_clases_fruto_lectura_geron_deep_learning/11_chapter/1_introduccion/1_introduccion.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

### [Evaluamos al profesor Marco Cañas Aquí](https://forms.office.com/Pages/ResponsePage.aspx?id=IefhmYRxjkmK_7KtTlPBwkanXIs1i1FEujpsZgO6dXpUREJPV1kxUk1JV1ozTFJIQVNIQjY5WEY3US4u)

Este texto es el punto de partida para dominar las redes neuronales profundas (Deep Learning). 


# Entrenamiento de Redes Neuronales Profundas


En el Capítulo 10, construiste, entrenaste y ajustaste varias redes neuronales artificiales usando PyTorch. 

Pero eran redes superficiales con solo unas pocas capas ocultas. 

¿Qué pasa si necesitas abordar un problema complejo, como detectar cientos de tipos de objetos en imágenes de alta resolución? 

Es posible que necesites entrenar una ANN mucho más profunda, tal vez con docenas o incluso cientos de capas, cada una con cientos de neuronas, vinculadas por cientos de miles de conexiones.



Entrenar una red neuronal profunda no es un paseo por el parque. 

Estos son algunos de los problemas con los que podrías encontrarte:  

* Podrías enfrentarte al problema de los **gradientes que se vuelven cada vez más pequeños o más grandes** al fluir hacia atrás a través de la DNN durante el entrenamiento. Ambos problemas hacen que las capas inferiores sean muy difíciles de entrenar.


* Es posible que **no tengas suficientes datos** de entrenamiento para una red tan grande, o que sea demasiado costoso etiquetarlos.


* El entrenamiento puede ser **extremadamente lento**.


* Un modelo con millones de parámetros corre el riesgo de sufrir un **severo sobreajuste (overfitting)** en el conjunto de entrenamiento, especialmente si no hay suficientes instancias de entrenamiento o si estas tienen mucho ruido.



En este capítulo, repasaremos cada uno de estos problemas y presentaremos diversas técnicas para resolverlos. 

Comenzaremos explorando los problemas de **desvanecimiento y explosión de gradientes** y algunas de sus soluciones más populares, incluyendo la:
* **inicialización inteligente de pesos, 
* mejores funciones de activación, 
* normalización por lotes (Batch-Norm), 
* normalización de capas (Layer-Norm) y 
* recorte de gradientes (Gradient Clipping)**.



Luego, veremos:
* el **aprendizaje por transferencia (Transfer Learning)** y el 
* **preentrenamiento no supervisado**, que pueden ayudarte a abordar tareas complejas incluso cuando tienes pocos datos etiquetados. 

Después, discutiremos una variedad de **optimizadores** que pueden acelerar enormemente el entrenamiento de modelos grandes. 

También discutiremos cómo puedes ajustar la **tasa de aprendizaje (Learning Rate)** durante el entrenamiento para acelerar el proceso y producir mejores modelos. 

Finalmente, cubriremos algunas técnicas de **regularización** populares: regularización $\ell_1$ y $\ell_2$, 
* **Dropout**, 
* **Monte Carlo Dropout** y 
* **regularización Max-Norm**.


## 2. Complemento de Conceptos Clave
Para que la introducción tenga sentido, aquí tienes una breve explicación técnica de lo que aprenderás:



* **Vanishing Gradients:** Ocurre cuando el gradiente se multiplica por valores pequeños en cada capa, llegando a ser casi cero en las primeras capas. La red deja de aprender.
* **Exploding Gradients:** Lo opuesto; los gradientes crecen exponencialmente, haciendo que los pesos oscilen violentamente y el modelo nunca converja.


* **Batch Normalization:** Técnica que escala las entradas de cada capa para que tengan media cero y varianza unitaria, estabilizando el aprendizaje.


* **Dropout:** Apaga neuronas aleatoriamente durante el entrenamiento para evitar que la red dependa demasiado de conexiones específicas (evita el overfitting).




## 3. Práctica en Python (PyTorch)
Vamos a crear un script que genere datos aleatorios para un problema de clasificación y entrene una red **profunda** aplicando algunas de las técnicas mencionadas (Inicialización He, Dropout y Batch Normalization).


<img src = 'red_neuronal_1.png'>

Aquí tienes la representación visual detallada de la red neuronal que definimos en el script de PyTorch. 

Geminis ha diseñado este diagrama para que actúe como una "radiografía" técnica del modelo DeepNet que vamos a consruir.  



# Guía de la Imagen  

Este diagrama traduce el código en componentes visuales:  

1. Entrada y Dimensiones (Extremo Izquierdo): Verás una columna de neuronas (nodos azules) que representan las 20 características (features) de tus datos generados aleatoriamente. Los números bajo los bloques (20, 128, 64, 32, 3) muestran exactamente cómo se expanden y contraen las dimensiones de los datos a medida que pasan por las capas.


2. Los Bloques Técnicos: En lugar de dibujar miles de conexiones individuales (que harían el diagrama ilegible), Geminis ha agrupado las operaciones de cada "bloque de capa":
  * Linear Layer (Capa Densa): Representa nn.Linear. Es el bloque blanco con la cuenta de neuronas.
  * Batch Norm (Normalización): El bloque gris claro es nn.BatchNorm1d, encargado de estabilizar los gradientes.
  * ReLU (Activación): El bloque verde es nn.ReLU. Notas que solo está presente en las capas ocultas, no en la salida.
  * Dropout (Regularización): El pequeño panel rojo es nn.Dropout(0.3), que "apaga" neuronas durante el entrenamiento.


3. La Salida (Extremo Derecho): Las últimas 3 neuronas (rojas) representan las 3 clases de tu problema de clasificación.
Esta imagen es el mapa de ruta perfecto para entender qué está pasando dentro de esta script DeepNet Architecture.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


In [ ]:

# 1. Generación de datos aleatorios (Sintéticos)
# Simulamos 1000 muestras, 20 características, 3 clases
X = torch.randn(1000, 20) # crea un tensor de 1000 filas y 20 columnas con valores aleatorios,
#con media 0 y desviación estándar 1. Es decir, crea datos que siguen una distribución normal estándar 
# y escalados con la técnica estandarización.
y = torch.randint(0, 3, (1000,)) 
X, y

In [ ]:

dataset = TensorDataset(X, y)
dataset 

In [ ]:
import pandas as pd 

df = pd.DataFrame(dataset.tensors[0].numpy(), columns=[f'feature_{i}' for i in range(20)])
df['label'] = dataset.tensors[1].numpy()
df.head()

In [ ]:
loader = DataLoader(dataset, batch_size=32, shuffle=True)  
# Crea un DataLoader que divide el dataset en lotes de 32 muestras y los mezcla aleatoriamente en cada época.
# DataLoader para manejar los datos en lotes. 
loader 

In [ ]:

# 2. Definición de una Red Neuronal Profunda (Deep Model)
class DeepNet(nn.Module):
    def __init__(self):
        super(DeepNet, self).__init__()
        # Usamos capas densas con Batch Normalization y Dropout
        self.layers = nn.Sequential(
            nn.Linear(20, 128),
            nn.BatchNorm1d(128), # Estabiliza gradientes
            nn.ReLU(), # Activa la no linealidad, permite aprender relaciones complejas, 
            # y ayuda a evitar el problema de gradientes desvanecientes.
            nn.Dropout(0.3),     # Evita overfitting
            # sigue la capa oculta con 128 neuronas, seguida de Batch Normalization, ReLU y Dropout.
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            # sigue la capa oculta con 64 neuronas, seguida de Batch Normalization y ReLU.
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 3)     # Capa de salida (3 clases)
        )
        
        # Inicialización de pesos (He Initialization para ReLU)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')

    def forward(self, x): 
        """Este método define cómo se propagan los datos a través de la red, es decir, 
        cómo se transforman las entradas en salidas utilizando las capas definidas en
        el constructor.""" 
        return self.layers(x)


In [ ]:

# 3. Configuración de entrenamiento
model = DeepNet()
criterion = nn.CrossEntropyLoss() 
# la clase CrossEntropyLoss combina LogSoftmax y NLLLoss, ideal para clasificación 
# multiclase
# Optimizador Adam (más rápido que SGD convencional)
optimizer = optim.Adam(model.parameters(), lr=0.001) 
# lr es la tasa de aprendizaje, controla qué tan rápido se actualizan los pesos 
# durante el entrenamiento. Un valor común es 0.001, pero puede ajustarse según el problema y 
# la arquitectura de la red.


In [ ]:
%%time 
# 4. Bucle de entrenamiento simple
print("Iniciando entrenamiento...")
for epoch in range(10):
    total_loss = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()           # Limpiar gradientes
        outputs = model(batch_X)        # Forward pass
        loss = criterion(outputs, batch_y) # Calcular error
        loss.backward()                 # Backward pass (Backpropagation) 
        # sirve para calcular los gradientes de la función de pérdida con respecto a los pesos de la red.
        optimizer.step()                # Actualizar pesos
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/10 - Loss: {total_loss/len(loader):.4f}")

print("\n¡Entrenamiento completado!")


## ¿Qué aplicamos en el código?
1.  **Batch Normalization:** Agregamos `nn.BatchNorm1d` para que los gradientes fluyan mejor.
2.  **Dropout:** Agregamos `nn.Dropout(0.3)` para que el modelo sea más robusto.
3.  **He Initialization:** Usamos `kaiming_normal_` para inicializar los pesos, lo cual es vital para que las funciones ReLU no "mueran" al inicio.
4.  **Optimización:** Usamos **Adam**, que adapta la tasa de aprendizaje automáticamente, resolviendo el problema de la lentitud mencionado en el texto.

In [ ]:
import torch

def verificar_pytorch():
    print(f"Versión de PyTorch: {torch.__version__}")
    
    # Comprobar si hay soporte para GPU (CUDA)
    if torch.cuda.is_available():
        print(f"¡Éxito! CUDA está disponible.")
        print(f"Dispositivo actual: {torch.cuda.get_device_name(0)}")
    else:
        print("PyTorch instalado correctamente, pero funcionando solo en CPU.")

if __name__ == "__main__":
    try:
        verificar_pytorch()
    except ImportError:
        print("Error: PyTorch sigue sin detectarse. Prueba reiniciar tu IDE o Kernel.")

Para dominar el Deep Learning, no basta con correr el script; hay que entender qué siente la red en cada paso. 

Aquí tienes un diseño de 4 actividades secuenciales para desmenuzar tu práctica de PyTorch:

---


# Diseño de 4 actividades secuenciales para desmenuzar la práctica de PyTorch:


## Actividad 1: El Origen de los Datos (Dataset)
**Objetivo:** Entender que la red solo ve números en una estructura específica.

* **Instrucción:** En tu código, después de crear `X` e `y`, imprime sus dimensiones y un ejemplo de los datos.
* **Código a ejecutar:**


In [ ]:
print(f"Forma de X: {X.shape}") # (1000, 20) -> 1000 ejemplos, 20 preguntas cada uno
print(f"Primer dato: {X[0]}")   # Verás 20 números aleatorios
print(f"Etiqueta del primer dato: {y[0]}") # La respuesta correcta (0, 1 o 2)


* **Concepto clave:** Los datos deben ser `Tensors` (el lenguaje de PyTorch) y estar organizados en filas (muestras) y columnas (características).




## Actividad 2: El "Cerebro" y sus Filtros (Arquitectura)
**Objetivo:** Visualizar cómo se transforman los datos.

* **Instrucción:** Pasa una sola muestra de datos por la red (sin entrenar) y observa la salida.
* **Código a ejecutar:**



In [ ]:
# 1. Ponemos el modelo en modo evaluación
model.eval() 


In [ ]:

# 2. Tomamos la muestra y le damos forma de lote (1, 20)
una_muestra = X[0].unsqueeze(0) # el método unsqueeze(0) agrega una dimensión adicional al tensor en la posición 0,
X[0].shape, una_muestra.shape # (1, 20), es decir, ahora tenemos un lote con una sola muestra, listo para ser procesado por la red.

In [ ]:

# 3. Usamos torch.no_grad() para no gastar memoria calculando gradientes
with torch.no_grad():
    prediccion_cruda = model(una_muestra)

print(f"Salida de la red (Logits): {prediccion_cruda}")


In [ ]:

# 4. (Opcional) Si quieres ver la probabilidad por clase:
probabilidades = torch.softmax(prediccion_cruda, dim=1)
print(f"Probabilidades: {probabilidades}")


* **Concepto clave:** La red toma 20 números y, tras pasar por las capas, devuelve 3 números. El número más alto indica la clase que la red "cree" que es la correcta.

---



## Actividad 3: Entendiendo la "Pérdida" (The Loss)
**Objetivo:** Comprender que la "Loss" es el termómetro del error.



* **Significado de la Pérdida (Loss):** Imagina que estás practicando tiro al blanco. La **Loss** es la distancia entre donde cayó tu flecha y el centro del blanco.
    * **Loss Alta:** Estás muy lejos de la verdad. La red está "confundida".
    * **Loss Baja:** Estás muy cerca de acertar. La red está aprendiendo patrones reales.
    * En clasificación, usamos `CrossEntropyLoss`, que penaliza mucho si la red está muy segura de una respuesta incorrecta. (es decir, penaliza mucho a los falsos positivos). 





* **Ejercicio mental:** Si la `Loss` no baja durante el entrenamiento, tu red no está aprendiendo; es como si tiraras flechas con los ojos vendados y nadie te dijera qué tan lejos caen.

---



## Actividad 4: Midiendo el Éxito (Desempeño)
**Objetivo:** Traducir la Loss a algo humano (Precisión).

* **Instrucción:** Al final de tu entrenamiento, calcula el `% de precisión` (Accuracy). La Loss es para que la red aprenda, la Accuracy es para que tú entiendas qué tan buena es.
* **Código a ejecutar:**
    ```python


In [ ]:
model.eval() # Ponemos la red en modo evaluación
with torch.no_grad(): # No calculamos gradientes (ahorramos memoria)
    outputs = model(X)
    _, predicted = torch.max(outputs, 1)
    correctos = (predicted == y).sum().item()
    accuracy = (correctos / y.size(0)) * 100
    print(f"Precisión final: {accuracy:.2f}%")


# Resumen del Proceso de Aprendizaje
| Paso | Acción | Analogía |
| :--- | :--- | :--- |
| **1. Dataset** | Preparar `X` e `y`. | Estudiar el temario del examen. |
| **2. Forward** | Pasar datos por la red. | Dar una respuesta en el examen. |
| **3. Loss** | Calcular el error. | El profesor califica qué tan lejos estuviste de la verdad. |
| **4. Optimizer** | Ajustar pesos. | Corregir tus errores para el próximo examen. |



¿Qué parte de la arquitectura (Batch Norm, Dropout, etc.) te genera más curiosidad sobre cómo afecta a la pérdida?

# Proyecto de aplicación de este tema en el contexto de Administración de Empresas

Para estudiantes de **Administración de Empresas** en la **UdeA (Sede Caucasia)**, el enfoque no debe ser la programación pura, sino la **toma de decisiones basada en datos** y la eficiencia operativa.



Caucasia es el corazón del Bajo Cauca, una zona con gran potencial en agroindustria, comercio y servicios mineros.

Este es un diseño de un proyecto aplicado.



# Proyecto: "Optimización Inteligente de Operaciones en el Bajo Cauca"



## 1. El Problema de Negocio
Las empresas locales (ej. una distribuidora de insumos agrícolas o una comercializadora de oro) enfrentan incertidumbre en la demanda y altos costos operativos. 

Los estudiantes deben usar una **Red Neuronal Profunda (DNN)** para predecir el **Riesgo de Fuga de Clientes (Churn)** o la **Demanda de Inventario**, factores que afectan directamente el flujo de caja.



## 2. Objetivos del Proyecto
* **Identificar** una variable crítica de negocio (Ventas, Créditos, o Deserción).
* **Diseñar** la arquitectura de una red neuronal que procese datos históricos.
* **Interpretar** los resultados (Loss y Accuracy) no como números, sino como indicadores de **Riesgo Financiero**.

---



## 3. Fases de Ejecución



### Fase A: Recolección y "Limpieza" de Datos
En lugar de datos aleatorios, los estudiantes trabajarán con un dataset simulado de una empresa local.
* **Entradas (Features):** 
    - Antigüedad del cliente, 
    - volumen de compras, 
    - frecuencia de visitas, 
    - retrasos en pagos, 
    - ubicación (Caucasia, El Bagre, Tarazá).
* **Salida (Target):** ¿El cliente dejará de comprar? (Si/No). (es decir, vamos a hacer clasificación binaria). 



### Fase B: Construcción del "Cerebro" de la Empresa
Usando el script de PyTorch que vimos anteriormente, los estudiantes ajustarán la red:
* **Batch Normalization:** Para manejar la alta varianza de los datos económicos de la región.
* **Dropout:** Para asegurar que el modelo sea robusto y no se aprenda "de memoria" a los clientes actuales (Overfitting).



### Fase C: Simulación de Estrategia Administrativa
Aquí es donde aplican su formación profesional. Deberán responder:
1.  **Si la pérdida (Loss) es alta:** 
     - "¿Por qué nuestro modelo es incapaz de entender al cliente de Caucasia? 
     - ¿Faltan datos de contexto social?".
2.  **Si la precisión es del 85%:** "
     - ¿Cuánto dinero ahorramos si contactamos proactivamente a ese 85% de clientes antes de que se vayan?".



## 4. Estructura Sugerida del Informe Final

| Sección | Contenido para el Administrador |
| :--- | :--- |
| **Diagnóstico** | Definición del KPI (Indicador Clave de Desempeño) a predecir. |
| **Arquitectura** | Justificación técnica: ¿Por qué usamos una red *profunda* y no un modelo simple? |
| **Análisis de Loss** | Relación entre el error del modelo y la pérdida monetaria potencial. |
| **Propuesta de Valor** | Plan de acción basado en las predicciones del modelo. |

---



## 5. Ejemplo de Aplicación Práctica (Python)


Para que veamos la utilidad real, puedemos usar este bloque de código para traducir la **Pérdida (Loss)** en **Impacto Económico**:


In [ ]:
# Supongamos que cada error del modelo le cuesta a la empresa $500.000 COP
costo_error = 500_000 
loss_final = 0.45 # Resultado del entrenamiento

impacto_financiero = loss_final * costo_error
print(f"Impacto estimado de la incertidumbre: ${impacto_financiero:,.0f} COP por operación")



### ¿Por qué este proyecto en Caucasia?
Este proyecto posiciona a los estudiantes de la **UdeA** no solo como administradores tradicionales, sino como **Gerentes de Datos**. 

En una región en transformación, la capacidad de predecir tendencias mediante *Deep Learning* les otorga una ventaja competitiva inmensa en el mercado laboral.

Una guía estructurada les permitirá perderle el miedo al código y enfocarse en la **estrategia empresarial**. 


# Guía del Estudiante: Inteligencia Artificial para la Toma de Decisiones
**Curso:** Introducción a la Gestión de Datos y Analítica  
**Contexto:** Optimización de la Retención de Clientes en el Bajo Cauca Antioqueño.

---



## 1. Introducción al Caso
* Ustedes son los nuevos Gerentes de Estrategia de **"Suministros del Cauca"**, una empresa que distribuye maquinaria y agroquímicos. 
* Últimamente, varios clientes clave han dejado de comprar. 
* Su misión es construir una **Red Neuronal Profunda** que analice los datos históricos y prediga qué clientes están en riesgo, permitiendo a la gerencia actuar antes de que se pierdan.



## 2. Paso a Paso del Proyecto

### Fase 1: Preparación del Terreno (Dataset)
Antes de programar, deben entender qué le preguntarán a la IA. Sus datos contienen:
* **Variables de entrada (X):** Frecuencia de pago, monto de la última compra, años de antigüedad y ubicación geográfica.
* **Variable de salida (y):** ¿Se fue de la empresa? (1 = Sí, 0 = No).

**Actividad:** Ejecuten el script de generación de datos y analicen las primeras 5 filas. ¿Qué historia cuentan esos números sobre un cliente de Caucasia?



### Fase 2: Configuración del "Consultor Digital" (La Red)
Usarán una red con **3 capas ocultas**. Para que el modelo sea profesional, incluiremos:
* **Batch Normalization:** Para que los datos no "mareen" a la red si hay mucha diferencia entre un pequeño agricultor y una gran mina.
* **Dropout:** Para que la red no se vuelva perezosa y aprenda a generalizar.





### Fase 3: Interpretación del Entrenamiento (Loss)
Durante el entrenamiento, verán el valor de **Loss**. 
* **Su tarea:** Observen cómo baja el número. Si el Loss se estanca en un valor alto, significa que las variables que eligieron no explican por qué el cliente se va. 
* **Pregunta de reflexión:** ¿Faltarán datos del clima o de la situación de orden público en la región para que la red entienda mejor el entorno?



### Fase 4: El Reporte Gerencial (Medida de Desempeño)
No le presentarán la "Loss" al dueño de la empresa; le presentarán la **Precisión (Accuracy)**.
* Si la precisión es del 90%, significa que de cada 10 clientes, la IA identifica correctamente a 9.

---



## 3. Script Base para la Práctica


Copien y peguen este código en su entorno (Jupyter o Google Colab). 

Este ya incluye la corrección del modo evaluación (`model.eval()`) para que no tengan errores al probar clientes individuales.



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- SIMULACIÓN DE DATOS DE "SUMINISTROS DEL CAUCA" ---
X = torch.randn(500, 10) # 500 clientes, 10 indicadores económicos
y = torch.randint(0, 2, (500,)) # 0: Se queda, 1: Se va

# --- ARQUITECTURA DE LA RED ---
class RedGerencial(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(10, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2) # Salida: 2 opciones (Se queda o Se va)
        )
    def forward(self, x): return self.red(x)

# --- ENTRENAMIENTO ---
model = RedGerencial()
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("Entrenando modelo de retención...")
for epoch in range(20):
    model.train()
    optimizer.zero_grad()
    out = model(X)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 5 == 0:
        print(f"Semana de análisis {epoch+1}: Error (Loss) = {loss.item():.4f}")


In [ ]:

# --- PRUEBA DE UN CLIENTE NUEVO ---
model.eval() 
cliente_nuevo = torch.randn(1, 10)
with torch.no_grad():
    pred = model(cliente_nuevo)
    riesgo = torch.argmax(pred).item()
    estado = "RIESGO DE FUGA" if riesgo == 1 else "CLIENTE LEAL"
    print(f"\nResultado para el cliente analizado: {estado}")



## 4. Entregable
Deben entregar un breve informe (2 páginas) respondiendo:
1.  **¿Cuál fue la Loss final?** Explique qué significa ese número para la estabilidad de la empresa.
2.  **Propuesta Administrativa:** Si el modelo detecta que 50 clientes se van a ir, ¿qué estrategia de mercadeo aplicarían en Caucasia para retenerlos?

---



### Nota para el docente:
Esta estructura fomenta el **Pensamiento Crítico**. Los estudiantes dejan de ver a la IA como algo de "ingenieros" y empiezan a verla como una herramienta de gestión para reducir pérdidas y mejorar la rentabilidad en su propia región.

# Manos a la obra a la realización de este proyecto  

Esta es una guía estructurada que permitirá perderle el miedo al código y enfocarse en la **estrategia empresarial**. 


# Guía del Estudiante: Inteligencia Artificial para la Toma de Decisiones
**Curso:** Introducción a la Gestión de Datos y Analítica  
**Contexto:** Optimización de la Retención de Clientes en el Bajo Cauca Antioqueño.

---

## 1. Introducción al Caso
Ustedes son los nuevos Gerentes de Estrategia de **"Suministros del Cauca"**, una empresa que distribuye maquinaria y agroquímicos. Últimamente, varios clientes clave han dejado de comprar. Su misión es construir una **Red Neuronal Profunda** que analice los datos históricos y prediga qué clientes están en riesgo, permitiendo a la gerencia actuar antes de que se pierdan.

---



## 2. Paso a Paso del Proyecto

### Fase 1: Preparación del Terreno (Dataset)
Antes de programar, deben entender qué le preguntarán a la IA. Sus datos contienen:
* **Variables de entrada (X):** Frecuencia de pago, monto de la última compra, años de antigüedad y ubicación geográfica.
* **Variable de salida (y):** ¿Se fue de la empresa? (1 = Sí, 0 = No).

**Actividad:** Ejecuten el script de generación de datos y analicen las primeras 5 filas. ¿Qué historia cuentan esos números sobre un cliente de Caucasia?


Para que tus estudiantes de Administración puedan trabajar con realismo, lo ideal es generar un **Pandas DataFrame**. Esto les permite ver encabezados de columnas (como "Monto_Compra" o "Ubicación") en lugar de solo matrices de números, facilitando la interpretación gerencial.



Aquí tienes el script para la **Fase 1**, diseñado para ser ejecutado en una celda de Jupyter o Google Colab.

---



## Script de la Fase 1: Generación de Datos de "Suministros del Cauca"



In [ ]:
import pandas as pd
import numpy as np
import torch

# 1. Configuración de la simulación
np.random.seed(42)
n_clientes = 500

# 2. Creación de variables de entrada (X)
data = {
    'Frecuencia_Pago_Dias': np.random.randint(15, 90, n_clientes), # Días entre pagos
    'Monto_Ultima_Compra_MCOP': np.random.uniform(1.5, 50.0, n_clientes), # Millones de COP
    'Antiguedad_Anios': np.random.uniform(1, 15, n_clientes),
    'Ubicacion_Codigo': np.random.randint(0, 4, n_clientes) # 0:Caucasia, 1:El Bagre, 2:Tarazá, 3:Nechí
}

df = pd.DataFrame(data)

# 3. Creación de la variable de salida (y) con una lógica de negocio
# Simulamos que si la frecuencia de pago es alta (tarda mucho en pagar) 
# y la antigüedad es baja, hay más riesgo de fuga.
riesgo = (df['Frecuencia_Pago_Dias'] * 0.6) - (df['Antiguedad_Anios'] * 2)
df['Se_Fue'] = (riesgo > np.percentile(riesgo, 70)).astype(int)

# 4. Mostrar las primeras 5 filas para el análisis de los estudiantes
print("--- PRIMERAS 5 FILAS DEL DATASET (VISTA GERENCIAL) ---")
df.head()


In [ ]:

# 5. Conversión a Tensores de PyTorch (Para la Fase 2)
X_tensor = torch.tensor(df.drop('Se_Fue', axis=1).values, dtype=torch.float32)
y_tensor = torch.tensor(df['Se_Fue'].values, dtype=torch.long)
X_tensor 

In [ ]:

print("\n--- FORMATO PARA LA IA ---")
print(f"Tensor X (Entradas): {X_tensor.shape}")
print(f"Tensor y (Salida): {y_tensor.shape}")


## Guía de Análisis para el Estudiante (Solución Fase 1)

Una vez que los estudiantes corran el código, deben realizar el siguiente ejercicio de interpretación. Supongamos que obtienen estos resultados en las primeras filas:

| Frecuencia_Pago_Dias | Monto_Ultima_Compra_MCOP | Antiguedad_Anios | Ubicacion_Codigo | **Se_Fue** |
| :--- | :--- | :--- | :--- | :--- |
| 80 | 1.8 | 1.2 | 0 (Caucasia) | **1** |
| 20 | 45.0 | 10.5 | 0 (Caucasia) | **0** |



### ¿Qué historia cuentan estos números?

Como administrador, el estudiante debería ser capaz de decir:

* **El Cliente 1 (Fila 0):** Es un cliente de Caucasia con **riesgo crítico**. Tarda mucho en pagar (80 días), compra montos pequeños y es muy nuevo (1.2 años). La IA predice que **se irá** (`Se_Fue = 1`). 
    * *Acción administrativa:* Llamada inmediata para ofrecer un descuento por pronto pago o asesoría técnica.


* **El Cliente 2 (Fila 1):** Es un cliente "estrella" de Caucasia. Paga rápido (cada 20 días), hace compras grandes (45 millones) y lleva una década con nosotros. La IA predice que **es leal** (`Se_Fue = 0`).
    * *Acción administrativa:* Programa de fidelización o crédito preferencial para maquinaria pesada.

---



### ¿Por qué hacerlo así?
Al ver nombres de columnas y valores en Pesos Colombianos o años, el estudiante de la **UdeA** deja de ver "X" y "Y" y empieza a ver **activos de la empresa**. 





**Siguiente paso sugerido:** Ahora que tienen los datos, pueden proceder a la **Fase 2** usando `X_tensor` y `y_tensor` para entrenar el modelo que definimos anteriormente. 


### Fase 2: Configuración del "Consultor Digital" (La Red)
Usarán una red con **3 capas ocultas**. Para que el modelo sea profesional, incluiremos:
* **Batch Normalization:** Para que los datos no "mareen" a la red si hay mucha diferencia entre un pequeño agricultor y una gran mina.
* **Dropout:** Para que la red no se vuelva perezosa y aprenda a generalizar.



Pasemos a la **Fase 2**. Aquí es donde construimos el "motor" que procesará los datos de los clientes de Caucasia. En términos administrativos, estamos diseñando las políticas de procesamiento de información de nuestro consultor digital.

---



## Script de la Fase 2: Construcción y Configuración de la Red

Este script toma los tensores `X_tensor` y `y_tensor` que creamos en la Fase 1 y define la arquitectura solicitada.


In [ ]:
import torch.nn as nn

# 1. Definición de la Arquitectura (El Cerebro del Consultor)
class ConsultorDigitalNet(nn.Module):
    def __init__(self, input_size):
        super(ConsultorDigitalNet, self).__init__()
        
        # Definimos 3 capas ocultas con técnicas de estabilización
        self.layers = nn.Sequential(
            # Capa Oculta 1
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),     # Normalización: equilibra grandes y pequeños clientes
            nn.ReLU(),
            nn.Dropout(0.2),        # Dropout: evita que el modelo sea "perezoso"
            
            # Capa Oculta 2
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Capa Oculta 3
            nn.Linear(32, 16),
            nn.ReLU(),
            
            # Capa de Salida (2 opciones: Leal o Fuga)
            nn.Linear(16, 2)
        )
        
    def forward(self, x):
        return self.layers(x)

# 2. Instanciar el modelo
# El tamaño de entrada es 4 (Frecuencia, Monto, Antigüedad, Ubicación)
input_dim = X_tensor.shape[1] 
model = ConsultorDigitalNet(input_dim)

print("--- ESTRUCTURA DE LA RED NEURONAL ---")
print(model)


## Guía de Análisis para el Estudiante (Conceptos Administrativos)

Es vital que el estudiante de la **UdeA** no vea solo código, sino herramientas de gestión. Explícales la arquitectura con estas analogías:



### A. ¿Por qué 3 Capas Ocultas?
* En administración, una decisión simple se toma con una regla (ej. "si debe más de 90 días, bloquéelo"). 
* Pero el comportamiento humano es complejo. 
* Las **capas adicionales** permiten que la red entienda interacciones sutiles, como: *"Este cliente de El Bagre compra poco, pero es muy antiguo y nunca se retrasa; trátelo como VIP"*.



### B. Batch Normalization (El "Ecualizador")


* En el Bajo Cauca tenemos clientes que facturan $1.5$ millones y otros que facturan $500$ millones. 
* Sin **Batch Normalization**, los clientes grandes "gritarían" tanto que la red ignoraría a los pequeños. 
* Esta capa normaliza los datos para que todos los perfiles sean escuchados con la misma importancia durante el aprendizaje.



### C. Dropout (Entrenamiento de Resiliencia)
* El **Dropout** apaga neuronas aleatoriamente. 
* Es como si en una oficina, un día falta el contador y otro día falta el gerente de ventas. 

* La organización (la red) debe ser capaz de funcionar y tomar buenas decisiones incluso si no tiene toda la información perfecta o si faltan algunos "expertos". 
* Esto hace que el modelo sea **robusto** y no se aprenda los datos de memoria (evita el Overfitting).


### Tarea para el estudiante:
Pida a sus estudiantes que observen la salida del comando `print(model)`. Deben identificar:
1.  ¿Cuántos parámetros (conexiones) creen que se están creando entre la capa de 64 y la de 32 neuronas?
2.  ¿Por qué la última capa tiene un `2` como valor de salida? (Respuesta esperada: Porque solo hay dos decisiones posibles: Fuga o Lealtad).




### Fase 3: Interpretación del Entrenamiento (Loss)
Durante el entrenamiento, verán el valor de **Loss**. 
* **Su tarea:** Observen cómo baja el número. Si el Loss se estanca en un valor alto, significa que las variables que eligieron no explican por qué el cliente se va. 
* **Pregunta de reflexión:** ¿Faltarán datos del clima o de la situación de orden público en la región para que la red entienda mejor el entorno?


Llegamos al momento de la verdad: **el entrenamiento**. 

En esta fase, el "Consultor Digital" va a intentar predecir el comportamiento de los clientes y nosotros, como administradores, evaluaremos qué tan bien está aprendiendo mediante la **Pérdida (Loss)**.

---



## Script de la Fase 3: Entrenamiento y Monitoreo del Error

En este bloque, configuramos el optimizador (el encargado de ajustar la estrategia) y el bucle de entrenamiento.


In [ ]:
import torch.optim as optim

# 1. Configuración de herramientas gerenciales
# Usamos Adam: un optimizador que ajusta la "velocidad de aprendizaje" automáticamente
optimizer = optim.Adam(model.parameters(), lr=0.01)
# Usamos CrossEntropyLoss: ideal para problemas de decisión (Sí/No)
criterion = nn.CrossEntropyLoss()

print("--- INICIANDO ENTRENAMIENTO DEL MODELO ---")

# Simulamos 50 "ciclos de estudio" (Epochs)
for epoch in range(50):
    model.train() # Modo entrenamiento activado
    
    # Reiniciar el proceso para un nuevo intento
    optimizer.zero_grad()
    
    # Realizar la predicción
    outputs = model(X_tensor)
    
    # Calcular el error (Loss)
    loss = criterion(outputs, y_tensor)
    
    # Backpropagation: La red analiza en qué se equivocó
    loss.backward()
    
    # Ajuste de pesos: La red corrige su criterio
    optimizer.step()
    
    # Reporte cada 10 ciclos
    if (epoch + 1) % 10 == 0:
        print(f"Ciclo {epoch + 1}: El error (Loss) es de {loss.item():.4f}")

print("\n--- ENTRENAMIENTO FINALIZADO ---")



## Guía de Análisis Gerencial: ¿Qué es la "Loss"?

Para un estudiante de Administración en Caucasia, la **Loss** no es un concepto matemático, es un **Indicador de Incertidumbre**.



### 1. El Significado del Número
* **Si la Loss baja:** El modelo está encontrando patrones. Está aprendiendo que, por ejemplo, los clientes de **Nechí** que compran menos de 5 millones tienen un perfil de riesgo específico.
* **Si la Loss se estanca (No baja más):** Hemos llegado al límite de lo que estos datos pueden explicar. 



### 2. Pregunta de Reflexión para el Aula
Observen el valor final de la Loss. Si no es cercana a cero, planteen lo siguiente:
> *"Muchachos, el modelo sabe cuánto compran y dónde viven, pero... ¿Acaso el modelo sabe si hubo un paro armado en el Bajo Cauca esa semana? ¿Sabe si el precio del gramo de oro cayó y por eso el cliente no tiene liquidez?"*



**Lección administrativa:** Ninguna IA es perfecta porque los datos a veces no capturan la realidad social y económica completa de una región tan compleja como la nuestra. Un administrador usa la IA como guía, pero aporta su **conocimiento del contexto** para tomar la decisión final.



## Dinámica en Clase: "El Diagnóstico del Consultor"

Pide a los estudiantes que anoten su Loss final. 
* **Grupo A (Loss < 0.3):** Tienen un consultor muy seguro de sus decisiones. 
* **Grupo B (Loss > 0.6):** Su consultor está "adivinando". 

**Reto:** ¿Qué variable adicional le pedirían al sistema de facturación de la empresa para bajar esa Loss? (Ejemplos: *¿Días de mora en el pasado? ¿Número de reclamos al servicio al cliente?*).

¿Procedemos a la **Fase 4** para ver cuántos clientes logramos salvar realmente y medir el éxito final del proyecto?


### Fase 4: El Reporte Gerencial (Medida de Desempeño)
No le presentarán la "Loss" al dueño de la empresa; le presentarán la **Precisión (Accuracy)**.
* Si la precisión es del 90%, significa que de cada 10 clientes, la IA identifica correctamente a 9.

---


Llegamos a la meta del proyecto. En esta **Fase 4**, traduciremos esos números técnicos (Loss) a una métrica que cualquier dueño de empresa o junta directiva entendería: la **Precisión (Accuracy)**. 



Como administradores de la **UdeA**, aquí es donde validamos si nuestra inversión en IA tiene sentido financiero.

---



## Script de la Fase 4: Evaluación de Desempeño Gerencial

En este bloque, verificamos qué tan acertadas son las predicciones del modelo frente a la realidad de los clientes.


In [ ]:
# 1. Cambiamos al modo de "Examen" (Evaluación)
model.eval()

# 2. Realizamos la predicción final sobre todos nuestros clientes
with torch.no_grad():
    predicciones_raw = model(X_tensor)
    # Elegimos la opción con mayor puntaje (0 o 1)
    _, clasificacion_final = torch.max(predicciones_raw, 1)

# 3. Calculamos el porcentaje de acierto (Accuracy)
aciertos = (clasificacion_final == y_tensor).sum().item()
total_clientes = y_tensor.size(0)
precision = (aciertos / total_clientes) * 100

print("--- REPORTE DE DESEMPEÑO FINAL ---")
print(f"Total de clientes analizados: {total_clientes}")
print(f"Clientes clasificados correctamente: {aciertos}")
print(f"Precisión del Modelo (Accuracy): {precision:.2f}%")

# 4. Impacto en el mundo real (Simulación)
fugas_detectadas = clasificacion_final.sum().item()
print(f"\nAlerta Gerencial: Se han detectado {fugas_detectadas} clientes con alto riesgo de fuga.")
print(f"Acción recomendada: Enviar equipo de fidelización a estas cuentas inmediatamente.")


## Guía de Análisis: De la Métrica a la Estrategia

Para cerrar el proyecto con sus estudiantes en Caucasia, utilice este marco de discusión:



## 1. La Matriz de Confusión (Concepto Clave)
No todos los errores son iguales en administración. Pídales que piensen en esto:
* **Falso Positivo:** La IA dice que un cliente se va a ir, pero en realidad es leal. 
    * *Costo:* Gastamos tiempo y un descuento innecesario en él.


* **Falso Negativo:** La IA dice que el cliente es leal, pero se va sin avisar.
    * *Costo:* Perdimos al cliente para siempre y su flujo de caja asociado.
    * **¿Cuál error es más caro para una empresa en el Bajo Cauca?** Generalmente, el Falso Negativo.



### 2. Conclusión del Proyecto
Para el informe final, los estudiantes deben redactar su **Propuesta de Valor**. Por ejemplo:
> *"Nuestro modelo alcanzó una precisión del 85%. Si cada cliente que se va representa una pérdida de $10$ millones de COP al año, identificar a tiempo a 40 de los 50 clientes en riesgo podría salvarle a 'Suministros del Cauca' hasta $400$ millones de COP anuales."*



---



## Cierre de la actividad: "IA con Sello UdeA"

Como última reflexión, recuérdeles que la IA no reemplaza al Administrador de Empresas de Caucasia; la IA es su **asistente analítico**. 

Un administrador que sabe interpretar una red neuronal es un profesional que puede liderar la transformación digital de la región, ya sea en el sector minero, ganadero o comercial.



## 3. Script Base para la Práctica
Copien y peguen este código en su entorno (Jupyter o Google Colab). 

Este ya incluye la corrección del modo evaluación (`model.eval()`) para que no tengan errores al probar clientes individuales.


Para cumplir con el diseño de un proyecto profesional y robusto, he modificado la clase `RedGerencial`. Ahora cuenta con **tres capas ocultas** de tamaño descendente (64, 32 y 16 neuronas), integrando en cada una **Batch Normalization** y **Dropout**, tal como lo discutimos para las fases anteriores.

Aquí tienes el script actualizado y listo para correr:


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

# Parámetros
num_features = 20
n = 1000

# --- 1. GENERACIÓN DE DATOS CON PATRÓN (Para que el Loss pueda bajar) ---
X = torch.randn(n, num_features)
# Creamos una relación lógica: si la suma de las primeras 5 variables es > 0, es clase 1
# Esto simula un comportamiento real que la red SI puede aprender.
y = (X[:, :5].sum(dim=1) > 0).long() 

# --- 2. ARQUITECTURA PROFUNDA ---
class RedGerencialOptimizada(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1), # Reducimos dropout ligeramente para facilitar convergencia
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 2)
        )
        
    def forward(self, x): 
        return self.red(x)

model = RedGerencialOptimizada()

# --- 3. OPTIMIZACIÓN AVANZADA ---
optimizer = optim.Adam(model.parameters(), lr=0.01)
# Scheduler: Reduce el LR si el loss deja de bajar
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
criterion = nn.CrossEntropyLoss()

print("--- Entrenando con Optimización Dinámica ---")

for epoch in range(100): # Subimos a 100 épocas
    model.train()
    optimizer.zero_grad()
    
    out = model(X)
    loss = criterion(out, y)
    
    loss.backward()
    optimizer.step()
    
    # Actualizamos el scheduler basado en el loss actual
    scheduler.step(loss)
    
    if (epoch + 1) % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Época {epoch+1:3d} | Loss: {loss.item():.4f} | LR: {current_lr:.4f}")
        
        if loss.item() < 0.3:
            print("¡Objetivo alcanzado! Loss menor a 0.3")
            break

# --- 4. VERIFICACIÓN FINAL ---
model.eval()
with torch.no_grad():
    logits = model(X)
    preds = torch.argmax(logits, dim=1)
    acc = (preds == y).float().mean()
    print(f"\nPrecisión Final: {acc*100:.2f}%")

--- Entrenando con Optimización Dinámica ---
Época  10 | Loss: 0.0396 | LR: 0.0100
¡Objetivo alcanzado! Loss menor a 0.3

Precisión Final: 99.30%


### Cambios realizados:
1.  **Profundidad:** Se añadieron capas intermedias para que la red pueda aprender relaciones más complejas entre las variables de negocio.
2.  **Jerarquía de neuronas:** La estructura `20 -> 64 -> 32 -> 16 -> 2` sigue un patrón de embudo que ayuda a sintetizar la información desde los datos brutos hasta la decisión final.
3.  **Consistencia técnica:** Se mantuvo el `BatchNorm1d` y el `Dropout` en las capas más grandes para asegurar que el modelo sea estable y no se sobreajuste a los datos de entrenamiento.

Con esta estructura, tus estudiantes estarán trabajando con una red neuronal de "grado industrial", similar a las que se usan en analítica de clientes real.

## 4. Entregable
Deben entregar un breve informe (2 páginas) respondiendo:
1.  **¿Cuál fue la Loss final?** Explique qué significa ese número para la estabilidad de la empresa.
2.  **Propuesta Administrativa:** Si el modelo detecta que 50 clientes se van a ir, ¿qué estrategia de mercadeo aplicarían en Caucasia para retenerlos?

---



### Nota para el docente:
Esta estructura fomenta el **Pensamiento Crítico**. Los estudiantes dejan de ver a la IA como algo de "ingenieros" y empiezan a verla como una herramienta de gestión para reducir pérdidas y mejorar la rentabilidad en su propia región.

### [Evaluamos al profesor Marco Cañas Aquí](https://forms.office.com/Pages/ResponsePage.aspx?id=IefhmYRxjkmK_7KtTlPBwkanXIs1i1FEujpsZgO6dXpUREJPV1kxUk1JV1ozTFJIQVNIQjY5WEY3US4u)

### Continue su aprendizaje en la siguiente clase a través del siguiente [vínculo]()